In [1]:
import matplotlib.pyplot as plt
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

# Load model
model_path = "C:/project_ours/real_fake_classifier_model1-20250220T182706Z-001/real_fake_classifier_model1"
loaded_model = tf.saved_model.load(model_path)
infer = loaded_model.signatures["serving_default"]

def classify_image(img):
    try:
        print("✅ Received Image Size:", img.size)
        img = img.resize((224, 224))
        img_array = np.array(img) / 255.0  # Normalize
        img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

        img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)

        input_key = list(infer.structured_input_signature[1].keys())[0]
        prediction = infer(**{input_key: img_tensor})
        prediction = list(prediction.values())[0].numpy()

        fake_prob = float(prediction[0][0])
        real_prob = 1 - fake_prob
        result = "Fake" if fake_prob > 0.5 else "Real"

        print(f"✅ Prediction: {result}, Fake Probability: {fake_prob:.4f}, Real Probability: {real_prob:.4f}")

        # Plot Bar Graph
        fig, ax = plt.subplots()
        ax.bar(["Real", "Fake"], [real_prob, fake_prob], color=['green', 'red'])
        ax.set_ylim([0, 1])
        ax.set_ylabel("Probability")
        ax.set_title("Confidence Score")

        return result, f"Fake: {fake_prob:.4%}, Real: {real_prob:.4%}", fig

    except Exception as e:
        print("❌ Error:", str(e))
        return "Error: " + str(e), None, None

# Gradio Interface
iface = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(type="pil", image_mode="RGB"),
    outputs=[gr.Text(), gr.Text(), gr.Plot()],  # Text for prediction & probability, Plot for bar chart
    title="Real vs Fake Image Classifier",
    description="Upload an image to classify it as Real or Fake. Displays confidence scores with a bar chart."
)

iface.launch()


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [4]:
import os
os.getcwd()

'C:\\Users\\Vedang Doley'